# Target Variables and Splits

Creates 12-month forward excess return targets (both regression and classification). Splits data into train (2005-2017), validation (2018-2021), and test (2022-2024) sets.


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from pandas.tseries.offsets import MonthEnd

In [5]:
# Paths & tag 
INTERIM_PATH  = Path("../data/interim")
PROCESSED_PATH = Path("../data/processed")
# "1b" for expanded universe
TAG = "1b"   

def _pick_returns_file(tag: str):
    cands = []
    if tag:
        cands += [INTERIM_PATH / f"returns_monthly_stocks_{tag}.parquet"]
    cands += [
        INTERIM_PATH / "returns_monthly_stocks.parquet"
    ]
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError(f"No returns_monthly_stocks parquet found for TAG='{tag}'.")

p = _pick_returns_file(TAG)
monthly_returns = pd.read_parquet(p)
monthly_returns["date"] = pd.to_datetime(monthly_returns["date"]) + MonthEnd(0)

print(f"Loaded: {p.name} | rows={len(monthly_returns):,} | "
      f"tickers={monthly_returns['ticker'].nunique()} | "
      f"span={monthly_returns['date'].min().date()}→{monthly_returns['date'].max().date()}")
monthly_returns.head()


Loaded: returns_monthly_stocks_1b.parquet | rows=24,129 | tickers=100 | span=2005-02-28→2025-09-30


,date,ticker,bench,l_stock,l_bench,ex_log,close,volume
0,2005-02-28,AAL.L,^FTSE,0.051415,0.023665,0.027750,1380.772461,21856477.0
1,2005-03-31,AAL.L,^FTSE,-0.032918,-0.015026,-0.017892,1336.060059,12819716.0
2,2005-04-30,AAL.L,^FTSE,-0.077620,-0.019122,-0.058499,1236.277466,27002956.0
3,2005-05-31,AAL.L,^FTSE,0.122271,0.033242,0.089029,1397.067749,44256497.0
4,2005-06-30,AAL.L,^FTSE,-0.002289,0.029614,-0.031903,1393.873657,13780426.0


## Classification Labels

Creates binary labels for whether a stock beats its benchmark over the next 12 months.


In [8]:
def forward_excess_target(df,
                          group_col="ticker",
                          date_col="date",
                          value_col="ex_log",
                          horizon=12):
    """
    Computes forward cumulative excess return over `horizon` months:
      ex_log_fwdH = cumsum(t+H) - cumsum(t)
      ex_simple_fwdH = expm1(ex_log_fwdH)
    Drops rows without a full future window.
    """
    df2 = df[[date_col, group_col, value_col]].copy()
    df2 = df2.sort_values([group_col, date_col])

    # Groupby should be on df2, not the outer df
    df2["cs"] = df2.groupby(group_col, sort=False)[value_col].cumsum()

    future_cs = df2.groupby(group_col, sort=False)["cs"].shift(-horizon)
    df2["excess_log_fwd12"]    = future_cs - df2["cs"]
    df2["excess_simple_fwd12"] = np.expm1(df2["excess_log_fwd12"])

    # drop rows without full window
    df2 = df2[df2["excess_log_fwd12"].notna()].reset_index(drop=True)
    return df2

stocks_monthly_targets_fwd12 = forward_excess_target(monthly_returns, horizon=12)
stocks_monthly_targets_fwd12.head()


,date,ticker,ex_log,cs,excess_log_fwd12,excess_simple_fwd12
0,2005-02-28,AAL.L,0.027750,0.027750,0.341746,0.407403
1,2005-03-31,AAL.L,-0.017892,0.009858,0.372080,0.450750
2,2005-04-30,AAL.L,-0.058499,-0.048641,0.472453,0.603924
3,2005-05-31,AAL.L,0.089029,0.040388,0.353708,0.424340
4,2005-06-30,AAL.L,-0.031903,0.008486,0.395923,0.485755


In [10]:
# Ensure datetime + basic checks
s = stocks_monthly_targets_fwd12
if not pd.api.types.is_datetime64_any_dtype(s["date"]):
    s["date"] = pd.to_datetime(s["date"], errors="raise")

missing = s["date"].isna().sum()
print("Dates OK" if missing == 0 else f"Warning: {missing} NaT in date")

print("Date range:", s["date"].min(), "→", s["date"].max())


Dates OK
Date range: 2005-02-28 00:00:00 → 2024-09-30 00:00:00


In [12]:
s = stocks_monthly_targets_fwd12
s["excess_log_fwd12"] = pd.to_numeric(s["excess_log_fwd12"], errors="coerce")

# Classification label: beat the market over the next 12 months?
s["label_win_12"] = (s["excess_log_fwd12"] > 0).astype("int8")

# (Optional) consistent names some later notebooks expect:
s = s.rename(columns={
    "excess_log_fwd12":    "Y_REG",        # regression target (log)
    "excess_simple_fwd12": "Y_REG_SIMPLE"  # regression target (simple)
})
# Keep classification as separate column
s["Y_CLS"] = s["label_win_12"].astype("int8")

# ADD 1-MONTH TARGETS FOR COMPARISON 
# Merge back with monthly_returns to get 1-month forward targets
monthly_returns_sorted = monthly_returns.sort_values(["ticker","date"]).copy()
g = monthly_returns_sorted.groupby("ticker")
l_stock_next = g["l_stock"].shift(-1)
l_bench_next = g["l_bench"].shift(-1)
monthly_returns_sorted["excess_log_next"] = l_stock_next - l_bench_next
monthly_returns_sorted["y_class_next"] = (monthly_returns_sorted["excess_log_next"] > 0).astype("int8")

# Merge 1-month targets into s
s = s.merge(
    monthly_returns_sorted[["date","ticker","excess_log_next","y_class_next"]],
    on=["date","ticker"],
    how="left"
)

# Rename 12-month classification for clarity
s = s.rename(columns={"Y_CLS": "y_up_12m"})

stocks_monthly_targets_fwd12 = s[["date","ticker","Y_REG","Y_REG_SIMPLE","y_up_12m","excess_log_next","y_class_next"]].copy()
stocks_monthly_targets_fwd12.head()


,date,ticker,Y_REG,Y_REG_SIMPLE,y_up_12m,excess_log_next,y_class_next
0,2005-02-28,AAL.L,0.341746,0.407403,1,-0.017892,0
1,2005-03-31,AAL.L,0.372080,0.450750,1,-0.058499,0
2,2005-04-30,AAL.L,0.472453,0.603924,1,0.089029,1
3,2005-05-31,AAL.L,0.353708,0.424340,1,-0.031903,0
4,2005-06-30,AAL.L,0.395923,0.485755,1,0.062843,1


In [14]:
train_end = pd.Timestamp("2017-12-31")
val_start, val_end   = pd.Timestamp("2018-01-01"), pd.Timestamp("2021-12-31")
test_start, test_end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-09-30")

df = (monthly_returns
      .merge(stocks_monthly_targets_fwd12, on=["date","ticker"], how="inner", validate="one_to_one")
      .sort_values(["ticker","date"])
      .reset_index(drop=True))

df["split"] = np.select(
    [
        df["date"] <= train_end,
        (df["date"] >= val_start) & (df["date"] <= val_end),
        (df["date"] >= test_start) & (df["date"] <= test_end),
    ],
    ["train", "val", "test"],
    default="drop"
)
df = df[df["split"] != "drop"].reset_index(drop=True)
df.head()


,date,ticker,bench,l_stock,l_bench,ex_log,close,volume,Y_REG,Y_REG_SIMPLE,y_up_12m,excess_log_next,y_class_next,split
0,2005-02-28,AAL.L,^FTSE,0.051415,0.023665,0.027750,1380.772461,21856477.0,0.341746,0.407403,1,-0.017892,0,train
1,2005-03-31,AAL.L,^FTSE,-0.032918,-0.015026,-0.017892,1336.060059,12819716.0,0.372080,0.450750,1,-0.058499,0,train
2,2005-04-30,AAL.L,^FTSE,-0.077620,-0.019122,-0.058499,1236.277466,27002956.0,0.472453,0.603924,1,0.089029,1,train
3,2005-05-31,AAL.L,^FTSE,0.122271,0.033242,0.089029,1397.067749,44256497.0,0.353708,0.424340,1,-0.031903,0,train
4,2005-06-30,AAL.L,^FTSE,-0.002289,0.029614,-0.031903,1393.873657,13780426.0,0.395923,0.485755,1,0.062843,1,train


In [16]:
mm = df.groupby("split")["date"].agg(["min","max","nunique"])
print(mm)

assert mm.loc["train","max"] <= train_end
assert val_start <= mm.loc["val","min"] <= val_end
assert val_start <= mm.loc["val","max"] <= val_end
assert test_start <= mm.loc["test","min"] <= test_end
assert test_start <= mm.loc["test","max"] <= test_end

print("NaNs in Y_REG:", df["Y_REG"].isna().sum())
# Check for 12-month target (backward compatibility)
if "y_up_12m" in df.columns:
    print("NaNs in y_up_12m (12m):", df["y_up_12m"].isna().sum())
elif "Y_CLS" in df.columns:
    print("NaNs in Y_CLS (12m, old name):", df["Y_CLS"].isna().sum())
else:
    print("Neither y_up_12m nor Y_CLS found in dataframe")
# Check for 1-month targets (for comparison)
if "excess_log_next" in df.columns:
    print("NaNs in excess_log_next (1m):", df["excess_log_next"].isna().sum())
if "y_class_next" in df.columns:
    print("NaNs in y_class_next (1m):", df["y_class_next"].isna().sum())

# Quick counts
print("\nRows by split:")
print(df["split"].value_counts())
print("\nTickers by split:")
print(df.groupby("split")["ticker"].nunique())


             min        max  nunique
split                               
test  2022-01-31 2024-09-30       33
train 2005-02-28 2017-12-31      155
val   2018-01-31 2021-12-31       48
NaNs in Y_REG: 0
NaNs in y_up_12m (12m): 0
NaNs in excess_log_next (1m): 0
NaNs in y_class_next (1m): 0

Rows by split:
split
train    14829
val       4800
test      3300
Name: count, dtype: int64

Tickers by split:
split
test     100
train    100
val      100
Name: ticker, dtype: int64


In [18]:
print(" Saving Processed Data ")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

def out(name: str, tag: str = TAG):
    stem, ext = name.rsplit(".", 1)
    return PROCESSED_PATH / f"{stem}{('_' + tag) if tag else ''}.{ext}"

# Save combined + per-split
df.to_parquet(out("stocks_monthly_targets_splits.parquet"), index=False)
df[df["split"]=="train"].to_parquet(out("train_set.parquet"), index=False)
df[df["split"]=="val"  ].to_parquet(out("val_set.parquet"), index=False)
df[df["split"]=="test" ].to_parquet(out("test_set.parquet"), index=False)

print("Saved:")
print(" ", out("stocks_monthly_targets_splits.parquet").name)
print(" ", out("train_set.parquet").name)
print(" ", out("val_set.parquet").name)
print(" ", out("test_set.parquet").name)
print(f"\nAll data saved to: {PROCESSED_PATH}")


 Saving Processed Data 
Saved:
  stocks_monthly_targets_splits_1b.parquet
  train_set_1b.parquet
  val_set_1b.parquet
  test_set_1b.parquet

All data saved to: ../data/processed
